# StreamFlix - Phase 3: KPI Calculations

This notebook calculates the ten required business KPIs and creates the optional cohort retention table.

In [1]:
import os
import pandas as pd

subscribers = pd.read_csv("subscribers.csv", parse_dates=["signup_date", "churn_date"])
titles = pd.read_csv("titles.csv")
watch_history = pd.read_csv("watch_history.csv", parse_dates=["watch_date"])
watchlist = pd.read_csv("watchlist.csv")

N_SUBSCRIBERS = len(subscribers)
N_ACTIVE = subscribers["is_active"].sum()
active_subscribers = subscribers[subscribers["is_active"]]

print(N_SUBSCRIBERS, "subscribers")
print(N_ACTIVE, "active subscribers")
print(len(watch_history), "watch sessions")


15000 subscribers
11199 active subscribers
650000 watch sessions


## KPI 1 - Total Watch Hours

In [2]:
total_watch_hours = watch_history["watch_duration_min"].sum() / 60
print("Total watch hours:", round(total_watch_hours))


Total watch hours: 3333468


## KPI 2 - Active Rate

In [3]:
active_rate = N_ACTIVE / N_SUBSCRIBERS * 100
print("Active rate:", round(active_rate, 1), "%")


Active rate: 74.7 %


## KPI 3 - Churn Rate

In [4]:
churn_rate = (N_SUBSCRIBERS - N_ACTIVE) / N_SUBSCRIBERS * 100
print("Churn rate:", round(churn_rate, 1), "%")
print(subscribers.groupby("plan_type")["is_active"].apply(lambda x: (1 - x.mean()) * 100).round(1))


Churn rate: 25.3 %
plan_type
Basic with Ads    34.3
Premium           18.1
Standard          24.1
Name: is_active, dtype: float64


## KPI 4 - Average Completion Rate

In [5]:
average_completion = watch_history["completion_pct"].mean()
print("Average completion:", round(average_completion, 1), "%")


Average completion: 65.3 %


## KPI 5 - Monthly Recurring Revenue

In [6]:
mrr = active_subscribers["monthly_price_usd"].sum()
print("MRR: $", round(mrr, 2))


MRR: $ 175868.51


## KPI 6 - ARPU

In [7]:
arpu = mrr / N_ACTIVE
print("ARPU: $", round(arpu, 2))


ARPU: $ 15.7


## KPI 7 - Average Watch Time per Active Subscriber

In [8]:
average_watch_time = total_watch_hours / N_ACTIVE
print("Average watch time:", round(average_watch_time, 1), "hours")


Average watch time: 297.7 hours


## KPI 8 - Watchlist Conversion

In [9]:
watchlist_conversion = watchlist["watched"].mean() * 100
print("Watchlist conversion:", round(watchlist_conversion, 1), "%")


Watchlist conversion: 46.2 %


## KPI 9 - Hit Concentration

In [10]:
plays_by_title = watch_history.groupby("title_id").size().sort_values(ascending=False)
top_title_count = max(1, round(len(plays_by_title) * 0.10))
hit_concentration = plays_by_title.head(top_title_count).sum() / plays_by_title.sum() * 100
print("Hit concentration:", round(hit_concentration, 1), "%")


Hit concentration: 31.0 %


## KPI 10 - Originals Share of Hours

In [11]:
watch_with_titles = watch_history.merge(
    titles[["title_id", "is_original"]],
    on="title_id",
    how="left"
)
originals_share = (
    watch_with_titles.loc[watch_with_titles["is_original"], "watch_duration_min"].sum()
    / watch_history["watch_duration_min"].sum()
    * 100
)
print("Originals share of hours:", round(originals_share, 1), "%")


Originals share of hours: 27.2 %


## KPI summary table

In [12]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Watch Hours", "Active Rate", "Churn Rate",
        "Average Completion Rate", "Monthly Recurring Revenue", "ARPU",
        "Average Watch Time per Active Subscriber", "Watchlist Conversion",
        "Hit Concentration", "Originals Share of Hours"
    ],
    "Value": [
        round(total_watch_hours), round(active_rate, 1), round(churn_rate, 1),
        round(average_completion, 1), round(mrr, 2), round(arpu, 2),
        round(average_watch_time, 1), round(watchlist_conversion, 1),
        round(hit_concentration, 1), round(originals_share, 1)
    ]
})
kpi_summary.to_csv("kpi_summary.csv", index=False)
kpi_summary.to_csv("kpi_summary_simple.csv", index=False)
kpi_summary


,KPI,Value
0,Total Watch Hours,3333468.00
1,Active Rate,74.70
2,Churn Rate,25.30
3,Average Completion Rate,65.30
4,Monthly Recurring Revenue,175868.51
5,ARPU,15.70
6,Average Watch Time per Active Subscriber,297.70
7,Watchlist Conversion,46.20
8,Hit Concentration,31.00
9,Originals Share of Hours,27.20


## Bonus - Cohort Retention

In [13]:
subscribers["signup_month"] = subscribers["signup_date"].dt.to_period("M")
subscribers["churn_month"] = subscribers["churn_date"].dt.to_period("M")

cohort_rows = []
for cohort, group in subscribers.groupby("signup_month"):
    row = {"signup_month": str(cohort), "subscribers": len(group)}
    for months in [3, 6, 12]:
        check_month = cohort + months
        retained = group["churn_month"].isna() | (group["churn_month"] > check_month)
        row[f"retained_{months}m_%"] = round(retained.mean() * 100, 1)
    cohort_rows.append(row)

cohort_retention = pd.DataFrame(cohort_rows)
cohort_retention.to_csv("cohort_retention.csv", index=False)
cohort_retention.to_csv("cohort_retention_simple.csv", index=False)
cohort_retention.head()


,signup_month,subscribers,retained_3m_%,retained_6m_%,retained_12m_%
0,2016-01,4,75.0,75.0,75.0
1,2016-02,9,88.9,88.9,88.9
2,2016-03,12,100.0,100.0,100.0
3,2016-04,13,100.0,100.0,100.0
4,2016-05,20,95.0,90.0,85.0


## Final results

In [14]:
print("Phase 3 complete")
print("KPI file: kpi_summary.csv")
print("Cohort file: cohort_retention.csv")


Phase 3 complete
KPI file: kpi_summary.csv
Cohort file: cohort_retention.csv


## Phase 3 Summary

- StreamFlix generated 3,333,468 total watch hours from 650,000 sessions. The active-subscriber rate is 74.7%, while the churn rate is 25.3%; churn is highest for Basic with Ads (34.3%) and lowest for Premium (18.1%).
- Engagement is substantial: average completion is 65.3%, average watch time is 297.7 hours per active subscriber, and watchlist conversion is 46.2%.
- The active subscriber base produces $175,868.51 in monthly recurring revenue with an ARPU of $15.70. Content consumption is moderately concentrated, with the top titles accounting for 31.0% of watch hours.
- Originals contribute 27.2% of total watch hours. The cohort table is also available for retention analysis, with the results saved to `cohort_retention.csv`; the complete KPI table is saved to `kpi_summary.csv`.
- Overall, the results indicate strong usage and monetization alongside a material churn opportunity, especially among Basic with Ads subscribers. Retention and plan-conversion initiatives should therefore be evaluated alongside content and engagement decisions.